# Inference Notebook

This notebook demonstrates how I run local inference with my fine-tuned Mistral code-fixing model exported in GGUF format. The goal is to evaluate the model in a realistic setting: given a buggy Python script (AI/ML project style), the model should return a corrected version that executes successfully with minimal changes.

# 1. Load the Model from Google Drive

In this step, I mount Google Drive to access the fine-tuned GGUF model file stored in Drive. I then install the minimal dependencies needed for the inference workflow and copy the GGUF file from Drive to the Colab local filesystem (`/content`). Running the model from local disk is typically faster and more reliable than reading directly from Drive, especially for large multi-GB files.

In [1]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!pip -q install -U langchain-openai langchain-core


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 3.2 MB/s eta 0:00:00


In [3]:
# 2) Copy the GGUF from Drive → Colab local disk (/content)
DRIVE_GGUF = "/content/gdrive/MyDrive/mistral-7b-instruct-v0.3.Q4_K_M.gguf"
LOCAL_GGUF = "/content/mistral_model.gguf"

!cp -v "$DRIVE_GGUF" "$LOCAL_GGUF"
!ls -lh "$LOCAL_GGUF"

'/content/gdrive/MyDrive/mistral-7b-instruct-v0.3.Q4_K_M.gguf' -> '/content/mistral_model.gguf'
-rw------- 1 root root 4.1G Jan 30 09:27 /content/mistral_model.gguf


# 2. Build `llama.cpp` with GPU Acceleration (CUDA)

In this section, I set up **llama.cpp**—the lightweight C/C++ runtime used to run GGUF models locally. I clone the repository and compile it with CUDA enabled so inference can be offloaded to the Colab GPU (A100), which significantly improves generation speed compared to CPU-only execution.

After building, I run a quick verification step to confirm that the compiled binaries include CUDA support and expose GPU-related options (such as GPU layer offloading).

In [4]:
%cd /content
!git clone https://github.com/ggerganov/llama.cpp
%cd /content/llama.cpp
!cmake -B build -DLLAMA_CUDA=ON
!cmake --build build -j


/content
Cloning into 'llama.cpp'...
remote: Enumerating objects: 77533, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (197/197), done.
remote: Total 77533 (delta 124), reused 34 (delta 34), pack-reused 77302 (from 4)
Receiving objects: 100% (77533/77533), 285.09 MiB | 47.48 MiB/s, done.
Resolving deltas: 100% (56035/56035), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
CMake Warning at CMakeLi

In [5]:
!ldd /content/llama.cpp/build/bin/llama-cli | grep -i cuda || true

	libggml-cuda.so.0 => /content/llama.cpp/build/bin/libggml-cuda.so.0 (0x0000783a14068000)
	libcudart.so.12 => /usr/local/cuda/targets/x86_64-linux/lib/libcudart.so.12 (0x0000783a13c00000)
	libcublas.so.12 => /usr/local/cuda/targets/x86_64-linux/lib/libcublas.so.12 (0x0000783a0d600000)
	libcuda.so.1 => /usr/lib64-nvidia/libcuda.so.1 (0x0000783a0ba00000)
	libcublasLt.so.12 => /usr/local/cuda/targets/x86_64-linux/lib/libcublasLt.so.12 (0x00007839ed400000)


In [6]:
!/content/llama.cpp/build/bin/llama-cli -h | grep -i -E "cuda|gpu|ngl|offload" | head

ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA A100-SXM4-40GB, compute capability 8.0, VMM: yes
-kvo,  --kv-offload, -nkvo, --no-kv-offload
                                        whether to enable KV cache offloading (default: enabled)
                                        (env: LLAMA_ARG_KV_OFFLOAD)
-dev,  --device <dev1,dev2,..>          comma-separated list of devices to use for offloading (none = don't
                                        offload)
-ngl,  --gpu-layers, --n-gpu-layers N   max. number of layers to store in VRAM, either an exact number,
                                        (env: LLAMA_ARG_N_GPU_LAYERS)
-sm,   --split-mode {none,layer,row}    how to split the model across multiple GPUs, one of:
                                        - none: use one GPU only
                                        - layer (default): split layers and KV across GPUs


In [46]:
# -----------------------
# Standard library
# -----------------------
import json
import random
import re
import subprocess
import time
from pathlib import Path

# -----------------------
# Third-party (HTTP)
# -----------------------
import requests

# -----------------------
# Third-party (LangChain)
# -----------------------
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage


# 3. Load the Dataset and Create a Reproducible Evaluation Split

Here I load my synthetic JSON dataset of code-fixing tasks and convert it into a Hugging Face `Dataset` object for easier indexing and sampling. To evaluate the model consistently, I create a fixed train/eval split using `train_test_split` with a deterministic seed (`seed=42`). This ensures that the same evaluation examples are selected every time I rerun this notebook.

In [10]:
DATA_PATH = "/content/final_dataset.json"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Samples:", len(data))
print("Keys:", list(data[0].keys()))


Samples: 582
Keys: ['title', 'description', 'difficulty', 'correct_code', 'incorrect_code', 'error_type']


In [12]:
from datasets import Dataset

dataset = Dataset.from_list(data)

split = dataset.train_test_split(test_size=0.15, seed=42)
train_dataset = split["train"]
eval_dataset  = split["test"]

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))


Train: 494
Eval : 88


# 4. Configure and Launch a Local `llama.cpp` Inference Server

In this section, I define the key file paths (dataset and GGUF model), then start a local llama.cpp server inside Colab. Running the model in server mode keeps it loaded (and optionally offloaded to GPU) so I can send multiple inference requests efficiently without reloading the model each time.

I also configure the main inference parameters—context length (CTX), maximum output tokens, temperature, and GPU layer offloading (`-ngl`)—and wait until the server is ready by polling the OpenAI-compatible `/v1/models` endpoint. Finally, I initialize a LangChain `ChatOpenAI` client pointing to this local server so later cells can send prompts and receive model outputs in a clean, structured way.

In [14]:
# -----------------------
# Paths (edit if needed)
# -----------------------
DATA_PATH  = "/content/final_dataset.json"
MODEL_GGUF = "/content/mistral_model.gguf"  # make sure GGUF is on /content for speed

# llama.cpp server binary (depends on your build; one of these usually exists)
SERVER_CANDIDATES = [
    "/content/llama.cpp/build/bin/llama-server",
    "/content/llama.cpp/build/bin/server",
]
SERVER_BIN = next((p for p in SERVER_CANDIDATES if subprocess.call(["bash","-lc", f"test -f {p}"]) == 0), None)
assert SERVER_BIN is not None, f"Couldn't find llama-server binary. Tried: {SERVER_CANDIDATES}"

# -----------------------
# Server + generation settings
# -----------------------
HOST = "127.0.0.1"
PORT = 8081
BASE_URL = f"http://{HOST}:{PORT}/v1"

CTX = 16384           # start with 4096; raise later if needed
MAX_TOKENS = 8000     # output length cap
TEMP = 0.0
NGL  = 99            # GPU layers (A100 can handle high offload)


In [42]:
# -----------------------
# Start llama-server once (keeps model in GPU memory)
# -----------------------
server_cmd = [
    SERVER_BIN,
    "-m", MODEL_GGUF,
    "--host", HOST,
    "--port", str(PORT),
    "-c", str(CTX),
    "-ngl", str(NGL),
]

print("Starting llama-server:\n", " ".join(server_cmd))
server_proc = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait until server is ready (poll /v1/models)
t0 = time.time()
ready = False
while time.time() - t0 < 180:  # up to 3 minutes
    try:
        r = requests.get(f"{BASE_URL}/models", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    # Print some server logs to help debug
    print("Server did not become ready. Last logs:")
    try:
        for _ in range(60):
            line = server_proc.stdout.readline()
            if not line:
                break
            print(line, end="")
    except Exception:
        pass
    raise RuntimeError("llama-server failed to start")

print("Server is ready.")


Starting llama-server:
 /content/llama.cpp/build/bin/llama-server -m /content/mistral_model.gguf --host 127.0.0.1 --port 8081 -c 16384 -ngl 99
Server is ready.


In [43]:
llm = ChatOpenAI(
    model="local-model",                         # llama-server accepts a model name field
    base_url= BASE_URL,         # llama-server OpenAI-compatible endpoint
    api_key="not-needed",                         # still required by client, but not used locally
    temperature=0.0,
    max_tokens=MAX_TOKENS,
)


# 5. Define the Inference Prompts (No-Hint and Retry-on-Error)

This cell defines the exact prompt templates used for inference and evaluation.

* `SYSTEM_TEXT` sets strict behavior for the model: it must act as a focused Python code fixer, apply minimal safe edits, avoid adding extra sections or unnecessary output, and return only runnable Python code.

* `build_user_prompt_no_hint` is the realistic inference prompt used in the first attempt. It provides the task context (title, difficulty, description) plus the buggy code, but does not provide any explicit error hint, which better reflects real-world usage.

* `build_user_prompt_with_error` is used only if the first attempt fails. It includes the captured runtime error message from executing the model’s output, guiding a second-pass correction that focuses on fixing the specific failure while still keeping changes minimal.

In [ ]:
# -----------------------
# Prompts
# -----------------------
SYSTEM_TEXT = (
    "You are a Python code fixer for AI/ML projects. "
    "Fix the provided code so it runs end-to-end and matches the task requirements. "
    "Rules: make the smallest possible change(s); do NOT add new features, demo blocks, or extra prints; "
    "do NOT refactor or rename unless required. "
    "Output ONLY the corrected Python code (no markdown, no explanations)."
)

def build_user_prompt_no_hint(ex):
    return (
        f"Task title: {ex['title']}\n"
        f"Difficulty: {ex['difficulty']}\n"
        f"Task description: {ex['description']}\n\n"
        "Buggy code:\n"
        "```python\n"
        f"{ex['incorrect_code']}\n"
        "```\n\n"
        "Fix the code. Output ONLY the corrected Python code."
    )

def build_user_prompt_with_error(ex, prev_code, error_text):
    return (
        f"Task title: {ex['title']}\n"
        f"Task description: {ex['description']}\n\n"
        "The following code still fails at runtime with this error:\n"
        f"{error_text}\n\n"
        "Code:\n"
        "```python\n"
        f"{prev_code}\n"
        "```\n\n"
        "Fix ONLY what is necessary to remove the error and satisfy the task. "
        "Do not add new sections. Output ONLY the corrected Python code."
    )


# 6. Helper Utilities for Cleaning, Inference Timing, and Runtime Validation

This section defines the core helper functions used throughout evaluation:

* `strip_code_fences` removes Markdown code fences (python ... ) in case the model returns formatted output, ensuring the result can be executed directly.

* `run_python` executes the model’s generated code in an isolated subprocess with a timeout. This provides a runtime-based correctness check (did the script crash or not?) and captures the last part of the error traceback for debugging and retry prompts.

* `llm_fix` sends a request to the local model via LangChain and records latency, allowing me to measure per-sample inference time and overall throughput.

* `parse_error_type_field` extracts the expected error type and (when available) the exact buggy line from the dataset’s error_type field. This lets me compare what the model produced against the known ground-truth bug.

* `extract_runtime_exception_name` attempts to identify the exception class (e.g., NameError, SyntaxError) from the captured traceback so I can categorize failures automatically.

* `contains_expected_bug_line` checks whether the model’s output still contains the dataset’s explicitly annotated buggy line—an easy way to detect cases where the model failed to modify the known error source.

In [ ]:
# -----------------------
# Helpers
# -----------------------
def strip_code_fences(code: str) -> str:
    code = (code or "").strip()
    code = re.sub(r"^\s*```(?:python)?\s*", "", code)
    code = re.sub(r"\s*```\s*$", "", code)
    return code.strip()

def run_python(code: str, timeout=25):
    """
    Run code in a subprocess. Returns (ok, stderr_tail).
    """
    code = strip_code_fences(code)
    tmp = Path("/content/_tmp_run_eval.py")
    tmp.write_text(code, encoding="utf-8")

    try:
        proc = subprocess.run(
            ["python3", str(tmp)],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.PIPE,
            text=True,
            timeout=timeout,
        )
        ok = (proc.returncode == 0)
        err = (proc.stderr or "")
        err_tail = "\n".join(err.splitlines()[-25:])
        return ok, err_tail
    except subprocess.TimeoutExpired:
        return False, f"TimeoutExpired: exceeded {timeout}s"
    except Exception as e:
        return False, f"RunnerError: {e}"

def llm_fix(messages):
    t0 = time.perf_counter()
    resp = llm.invoke(messages)
    t1 = time.perf_counter()
    return resp.content, (t1 - t0)

# ---- new: parse dataset error_type ----
def parse_error_type_field(error_type_text: str):
    """
    Your field looks like:
      "NameError: ... || line: y_pred = best_modell.predict(X_test)"
    We extract:
      expected_exception = "NameError"
      expected_line = "y_pred = best_modell.predict(X_test)"   (if present)
    """
    if not error_type_text:
        return None, None

    expected_exception = None
    expected_line = None

    # Exception name up to first colon
    m = re.match(r"\s*([A-Za-z_][A-Za-z0-9_]*)\s*:", error_type_text)
    if m:
        expected_exception = m.group(1)

    # Pull everything after "|| line:" if present
    m2 = re.search(r"\|\|\s*line:\s*(.*)\s*$", error_type_text)
    if m2:
        expected_line = m2.group(1).strip()

    return expected_exception, expected_line

# ---- new: parse runtime traceback to exception type ----
def extract_runtime_exception_name(stderr_tail: str):
    """
    Tries to find the last 'XError: message' line in stderr tail.
    """
    if not stderr_tail:
        return None
    # Look for patterns like "NameError: ..." or "IndexError: ..."
    matches = re.findall(r"([A-Za-z_][A-Za-z0-9_]*Error)\s*:\s*", stderr_tail)
    return matches[-1] if matches else None

def contains_expected_bug_line(code: str, expected_line: str):
    if not expected_line:
        return None  # unknown
    code = strip_code_fences(code)
    return expected_line.strip() in code


# 7. Run a Full Evaluation Loop on the Evaluation Split (with Automatic Retry)

This cell runs the main evaluation pipeline over the entire evaluation split (88 samples). For each example, I:

1. Sample evaluation indices from the eval set.

2. Extract the expected bug type and buggy line from the dataset’s error_type field (when available).

3. Attempt 1 (no hint): prompt the model using only the task context and the buggy code, then execute the returned code to check whether it runs without errors.

4. Attempt 2 (retry-on-error): if the first attempt fails, I capture the runtime traceback and send it back to the model as feedback to produce a corrected revision.

5. Track detailed metrics per sample, including:

    * whether the code ran successfully,

    * whether a retry was needed,

    * inference latency,

    * the runtime exception types observed,

    * and whether the model still left the explicitly annotated buggy line unchanged.

At the end, the notebook prints a concise summary (runtime success rate, retry usage, and average latency) to quantify the model’s practical bug-fixing performance.

In [44]:
sample_indices = random.sample(range(len(eval_dataset)), 88)

# -----------------------
# Main evaluation loop
# -----------------------
final_predictions = []
attempt_used = []
latencies = []
runtime_ok = []
runtime_errs = []

# NEW tracking
expected_exceptions = []
expected_lines = []
err1_types = []
err2_types = []
still_has_bug_line_after_1 = []
still_has_bug_line_after_2 = []
fixed_expected_bug_after_2 = []  # heuristic

for pos, idx in enumerate(sample_indices):
    ex = eval_dataset[pos]

    exp_exc, exp_line = parse_error_type_field(ex.get("error_type", ""))
    expected_exceptions.append(exp_exc)
    expected_lines.append(exp_line)

    # ---- Attempt 1 (no hint)
    msgs1 = [
        SystemMessage(content=SYSTEM_TEXT),
        HumanMessage(content=build_user_prompt_no_hint(ex)),
    ]
    pred1, t_llm1 = llm_fix(msgs1)
    pred1_clean = strip_code_fences(pred1)

    ok1, err1 = run_python(pred1_clean, timeout=25)
    err1_type = extract_runtime_exception_name(err1)
    err1_types.append(err1_type)

    still1 = contains_expected_bug_line(pred1_clean, exp_line)
    still_has_bug_line_after_1.append(still1)

    total_llm_time = t_llm1

    if ok1:
        final_predictions.append(pred1_clean)
        attempt_used.append(1)
        latencies.append(total_llm_time)
        runtime_ok.append(True)
        runtime_errs.append("")
        err2_types.append(None)
        still_has_bug_line_after_2.append(None)
        fixed_expected_bug_after_2.append(True)  # ran => bug not present (at least at runtime)
        print(f"[OK] pos={pos} | attempt=1 | llm_time={total_llm_time:.2f}s | expected={exp_exc}")
        continue

    # ---- Attempt 2 (with runtime error feedback)
    msgs2 = [
        SystemMessage(content=SYSTEM_TEXT),
        HumanMessage(content=build_user_prompt_with_error(ex, pred1_clean, err1)),
    ]
    pred2, t_llm2 = llm_fix(msgs2)
    pred2_clean = strip_code_fences(pred2)
    total_llm_time += t_llm2

    ok2, err2 = run_python(pred2_clean, timeout=25)
    err2_type = extract_runtime_exception_name(err2)
    err2_types.append(err2_type)

    still2 = contains_expected_bug_line(pred2_clean, exp_line)
    still_has_bug_line_after_2.append(still2)

    # heuristic: did we fix the expected bug?
    # - If it runs => yes
    # - If it fails but NOT with expected exception and the expected buggy line is gone => maybe fixed expected, but another bug exists
    if ok2:
        fixed_flag = True
    else:
        fixed_flag = (still2 is False) and (err2_type != exp_exc)
    fixed_expected_bug_after_2.append(fixed_flag)

    final_predictions.append(pred2_clean)
    attempt_used.append(2)
    latencies.append(total_llm_time)
    runtime_ok.append(ok2)
    runtime_errs.append("" if ok2 else err2)

    status = "OK" if ok2 else "FAIL"
    print(
        f"[{status}] pos={pos} | attempt=2 | llm_time={total_llm_time:.2f}s "
        f"| expected={exp_exc} | err1={err1_type} | err2={err2_type} | still_bug_line={still2}"
    )

print("\n=== Summary ===")
print("Total samples:", len(sample_indices))
print("Runtime success rate:", round(sum(runtime_ok) / len(runtime_ok) * 100, 2), "%")
print("Used attempt=1:", attempt_used.count(1))
print("Used attempt=2:", attempt_used.count(2))
print("Avg LLM time:", round(sum(latencies)/len(latencies), 2), "sec")

# Optional: quick insight
same_as_expected = sum((e is not None and e == r) for e, r in zip(expected_exceptions, err2_types) if r is not None)
print("Failures whose final error matches expected error_type:", same_as_expected)


[OK] pos=0 | attempt=1 | llm_time=14.18s | expected=NameError
[FAIL] pos=1 | attempt=2 | llm_time=28.57s | expected=NameError | err1=None | err2=None | still_bug_line=True
[OK] pos=2 | attempt=1 | llm_time=8.86s | expected=LogicError
[OK] pos=3 | attempt=1 | llm_time=9.83s | expected=SyntaxError
[FAIL] pos=4 | attempt=2 | llm_time=37.30s | expected=KeyError | err1=KeyError | err2=KeyError | still_bug_line=True
[OK] pos=5 | attempt=1 | llm_time=6.85s | expected=NameError
[OK] pos=6 | attempt=1 | llm_time=8.11s | expected=ValueError
[OK] pos=7 | attempt=1 | llm_time=5.56s | expected=SyntaxError
[OK] pos=8 | attempt=1 | llm_time=15.24s | expected=IndexError
[OK] pos=9 | attempt=1 | llm_time=15.80s | expected=ImportError
[OK] pos=10 | attempt=1 | llm_time=6.21s | expected=SyntaxError
[OK] pos=11 | attempt=2 | llm_time=15.13s | expected=AttributeError | err1=AttributeError | err2=None | still_bug_line=False
[FAIL] pos=12 | attempt=2 | llm_time=24.88s | expected=ValueError | err1=None | err2

# 8. Evaluation Analysis and Conclusion

This evaluation was performed on the full 88-sample evaluation split using a realistic two-stage inference strategy:

* Attempt 1: the model receives only the task context and buggy code (no explicit hint).

* Attempt 2: if the first attempt fails, the captured runtime error is provided back to the model to guide a second correction.

**Key Results (Observed)**

* Runtime success rate: ~56.8% (about half of the scripts ran end-to-end without crashing).

* Retry usage: the model required a second attempt for roughly half of the samples, but the retry step improved only a small portion of failures.

* Precision issue: in many failing cases, the output still contained the dataset’s explicitly annotated buggy line (`still_bug_line=True`) and often reproduced the same failure type. This indicates the model frequently misses the exact root cause even when the bug is small (e.g., a typo, wrong variable name, incorrect index, or a single missing character).

* Latency: average inference time was non-trivial, especially when a second attempt was required, which reduces practicality for larger-scale evaluation.

**Interpretation**

While the model can fix a meaningful portion of samples—especially straightforward syntax/runtime issues—its overall behavior is not consistently precise. The failure patterns suggest that the model sometimes:

* overlooks the exact faulty line,

* makes changes that are too conservative (leaving the bug untouched), or

* fails to correctly apply error feedback during the retry step.

For a code-fixing model where many bugs are small and localized, this level of reliability is not sufficient.

# Conclusion and Next Step

Based on these results, the current fine-tuned model is not accurate enough for dependable code correction on this dataset. The next step is to move to a stronger, more code-specialized base model (newer and higher-performing) and fine-tune again using the same dataset and evaluation pipeline, with the goal of achieving a substantially higher runtime pass rate and better precision on small bug patterns.